# ENG-209, Partie 1, Série 4

## pandas

1. Importez le package pandas. Si cela pose problème, faites `pip install pandas` dans un terminal.

In [1]:
import pandas as pd

# Set pandas to display all columns
pd.set_option('display.max_columns', None)

Lisez les données covid suisses dans une DataFrame _df_covid_.

Les données sont disponibles sur https://raw.githubusercontent.com/openZH/covid_19/master/COVID19_Fallzahlen_CH_total_v2.csv
* Trouvez comment n'importer que les colonnes suivantes pour ne pas gaspiller de l'espace mémoire avec ce qui ne nous intéresse pas: _date, abbreviation_canton_and_fl, ncumul_conf, current_hosp, ncumul_deceased_
* Utilisez la colonne date au format `datetime64[ns]` comme index

Astuce: L'option `parse_dates=True` peut être utilisée pour convertir toutes les colonnes qui ressemblent à une date en un type approprié.

In [14]:
df_covid = pd.read_csv(
    'https://raw.githubusercontent.com/openZH/covid_19/master/COVID19_Fallzahlen_CH_total_v2.csv',
    usecols=['date', 'abbreviation_canton_and_fl', 'ncumul_conf', 'current_hosp', 'ncumul_deceased'],
    parse_dates=True,
    index_col='date')
df_covid.reset_index().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24440 entries, 0 to 24439
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   date                        24440 non-null  datetime64[ns]
 1   abbreviation_canton_and_fl  24440 non-null  object        
 2   ncumul_conf                 23883 non-null  float64       
 3   current_hosp                21299 non-null  float64       
 4   ncumul_deceased             22141 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 954.8+ KB


2. Affichez les 5 première lignes de _df_covid_

In [3]:
df_covid.head()
# ou
df_covid[:5]

,abbreviation_canton_and_fl,ncumul_conf,current_hosp,ncumul_deceased
date,,,,
2020-02-01,GE,0.0,NaN,0.0
2020-02-02,GE,0.0,NaN,0.0
2020-02-03,GE,0.0,NaN,0.0
2020-02-04,GE,0.0,NaN,0.0
2020-02-05,GE,0.0,NaN,0.0


3. Affichez la forme (nombre d'éléments par dimension) de _df_covid_

In [4]:
df_covid.shape

(24440, 4)

4. Affichez les détails (nom des colonnes et leur types) de la structure de la DataFrame _df_covid_

In [5]:
df_covid.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24440 entries, 2020-02-01 to 2024-05-05
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   abbreviation_canton_and_fl  24440 non-null  object 
 1   ncumul_conf                 23883 non-null  float64
 2   current_hosp                21299 non-null  float64
 3   ncumul_deceased             22141 non-null  float64
dtypes: float64(3), object(1)
memory usage: 954.7+ KB


5. Afficher les statistiques sommaires de _df_covid_

In [6]:
df_covid.describe()

,ncumul_conf,current_hosp,ncumul_deceased
count,23883.000000,21299.000000,22141.000000
mean,61529.111837,47.060848,354.156678
std,115267.195530,79.793566,415.593993
min,0.000000,0.000000,0.000000
25%,2425.500000,4.000000,49.000000
50%,17841.000000,17.000000,189.000000
75%,77220.000000,51.000000,496.000000
max,803198.000000,669.000000,1890.000000


6. Affichez l'occupation mémoire de _df_covid_ et comparez-la à la taille du fichier CSV source

In [7]:
print(f"In memory: {df_covid.memory_usage(index=True, deep=True).sum() // 1024} kB")
# on disk with all data: about 2877 kB

In memory: 1980 kB


7. Avec une fonction _pandas_ à trouver en cherchant dans la documentation, trouvez le nombre de cantons différents dans _df_covid_. Pourquoi n'est-ce pas 26?

In [8]:
df_covid.abbreviation_canton_and_fl.drop_duplicates()

# Pas 26, car il y a FL dans la liste (Fürstentum Liechtenstein)

date
2020-02-01    GE
2020-02-24    BL
2020-02-24    JU
2020-02-25    TI
2020-02-26    GR
2020-02-26    AG
2020-02-27    BE
2020-02-27    FL
2020-02-27    LU
2020-02-27    ZH
2020-02-27    BS
2020-02-28    VD
2020-02-28    VS
2020-02-29    FR
2020-02-29    ZG
2020-03-01    NE
2020-03-02    SG
2020-03-03    SZ
2020-03-05    AR
2020-03-05    TG
2020-03-06    SO
2020-03-10    GL
2020-03-11    NW
2020-03-11    SH
2020-03-12    UR
2020-03-13    OW
2020-03-14    AI
Name: abbreviation_canton_and_fl, dtype: object

8. Créez une dataframe _df_vaud_ qui ne contient que les données du canton de Vaud, qui ne contient pas la colonne _abbreviation_canton_and_fl_ (inutile puisqu'on sait que c'est Vaud) et qui est triée par ordre décroissant de l'occupation hospitalière (_current_hosp_). Affichez les 5 premières lignes des colonnes _date_ (qui est l'index) et _current_hosp_.

In [9]:
df_vaud = (
    df_covid[df_covid.abbreviation_canton_and_fl == 'VD']
    .drop(columns='abbreviation_canton_and_fl')
    .sort_values(by='current_hosp', ascending=False)
)
df_vaud["current_hosp"].head()

date
2020-11-16    669.0
2020-11-17    659.0
2020-11-15    651.0
2020-11-18    641.0
2020-11-22    638.0
Name: current_hosp, dtype: float64

## Fonctions et lambdas

In [10]:
from typing import Callable

def filter(items: list, pred: Callable) -> list:
    filtered: list = []
    for item in items:
        if pred(item):
            filtered.append(item)
    return filtered

def is_odd(x: int) -> bool:
    return x % 2 == 1

print(filter([0, 1, 3, 3, 4, 5], is_odd))

pushkin = [
    "Онегин, добрый мой приятель,",
    "Родился на брегах Невы,",
    "Где, может быть, родились вы",
    "Или блистали, мой читатель;",
    "Там некогда гулял и я:",
    "Но вреден север для меня",
]

pushkin = [word for verse in pushkin for word in verse.split()]

def contains_e(s: str) -> bool:
    return 'е' in s

filtered = filter(pushkin, contains_e)
print(filtered)
print(filter(pushkin, lambda s: 'е' in s))


def map(items: list, f: Callable) -> list:
    mapped: list = []
    for item in items:
        mapped.append(f(item))
    return mapped

print(map(filtered, lambda s: len(s)))
print(map(filtered, len))

[1, 3, 3, 5]
['Онегин,', 'приятель,', 'брегах', 'Невы,', 'Где,', 'может', 'читатель;', 'некогда', 'вреден', 'север', 'меня']
['Онегин,', 'приятель,', 'брегах', 'Невы,', 'Где,', 'может', 'читатель;', 'некогда', 'вреден', 'север', 'меня']
[7, 9, 6, 5, 4, 5, 9, 7, 6, 5, 4]
[7, 9, 6, 5, 4, 5, 9, 7, 6, 5, 4]


Le corrigé ci-dessus n'utilise pas les types génériques. Voici une version avec les types génériques, que nous n'avons pas vus en cours, mais qui sont utilisables ainsi depuis Python 3.13. Cela donne du code plus élégant parce que plus rigoureux au niveau des types.

In [11]:
from typing import Callable

def filter[T](items: list[T], pred: Callable[[T], bool]) -> list[T]:
    filtered: list[T] = []
    for item in items:
        if pred(item):
            filtered.append(item)
    return filtered

def is_odd(x: int) -> bool:
    return x % 2 == 1

print(filter([0, 1, 3, 3, 4, 5], is_odd))

pushkin = [
    "Онегин, добрый мой приятель,",
    "Родился на брегах Невы,",
    "Где, может быть, родились вы",
    "Или блистали, мой читатель;",
    "Там некогда гулял и я:",
    "Но вреден север для меня",
]

pushkin = [word for verse in pushkin for word in verse.split()]

def contains_e(s: str) -> bool:
    return 'е' in s

filtered = filter(pushkin, contains_e)
print(filtered)
print(filter(pushkin, lambda s: 'е' in s))


def map[T, U](items: list[T], f: Callable[[T], U]) -> list[U]:
    mapped: list[U] = []
    for item in items:
        mapped.append(f(item))
    return mapped

print(map(filtered, lambda s: len(s)))
print(map(filtered, len))

[1, 3, 3, 5]
['Онегин,', 'приятель,', 'брегах', 'Невы,', 'Где,', 'может', 'читатель;', 'некогда', 'вреден', 'север', 'меня']
['Онегин,', 'приятель,', 'брегах', 'Невы,', 'Где,', 'может', 'читатель;', 'некогда', 'вреден', 'север', 'меня']
[7, 9, 6, 5, 4, 5, 9, 7, 6, 5, 4]
[7, 9, 6, 5, 4, 5, 9, 7, 6, 5, 4]
